# **Assignment 2 — Simplex Algorithm (Feasible Point → Vertex → Optimum)**  
### **Piyush Anand**  
**Roll No:** CS25MTECH12009  

---

## **Assumptions**
1. **Initial feasible point is given.**  
   The algorithm will first move this point to the nearest **vertex** (basic feasible solution),  
   and then perform the **vertex-to-vertex** simplex iterations until the optimal vertex is reached.

2. **Rank(A) = n.**  
   The constraint matrix A has full column rank, ensuring a unique vertex for every valid basis.

---

### **Steps Implemented**
1. **Phase I:** `feasible_to_vertex()` → moves from a feasible interior/boundary point to a vertex.  
2. **Phase II:** `geometric_simplex_trace()` → performs the geometric simplex (Assignment 1 method).  


In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import null_space


##  Read Input Data

The CSV file format remains the same as Assignment 1:

| Row | Description |
|------|--------------|
| 1 | Initial feasible point (`z₀`) |
| 2 | Cost vector (`c`) |
| 3 → end | Constraint matrix rows (`A`) with RHS values (`b`) |


In [ ]:
def read_input(filename):
    data = pd.read_csv(filename, header=None).values
    z0 = data[0, :-1]
    c  = data[1, :-1]
    A  = data[2:, :-1]
    b  = data[2:, -1]
    return np.array(A, float), np.array(b, float), np.array(c, float), np.array(z0, float)


## Phase I — Move from Feasible Point to Vertex

This function:
1. Finds the currently **tight constraints** (where equality holds).  
2. Computes a **null-space direction** that keeps current tight constraints satisfied.  
3. Moves along that direction until a new constraint becomes tight.  
4. Repeats until `rank(A_tight) = n`, i.e., we have a vertex.  


In [ ]:
from scipy.linalg import null_space
import numpy as np

def find_tight_rows(A, z, b, tol=1e-9):
    """Return mask of tight constraints and split A into tight/untight."""
    diff = A @ z - b
    tight_mask = np.abs(diff) < tol
    A1 = A[tight_mask]
    A2 = A[~tight_mask]
    return tight_mask, A1, A2


def feasible_to_vertex(A, b, z, c, n, tol=1e-9, epi=1e-3):
    """
    Phase I:
    Move from feasible point to vertex by following null-space directions
    until rank(A_tight) = n.
    """
    print("Phase I — Moving from feasible point to vertex...")
    track_cost, track_z = [float(c @ z)], [z.copy()]
    mask, A1, A2 = find_tight_rows(A, z, b, tol)
    rank = np.linalg.matrix_rank(A1) if len(A1) else 0
    if rank == n:
        print("Already at a vertex!\n")
        return z, track_cost, track_z, True
    iteration = 0
    bounded = True
    while rank < n:
        iteration += 1
        print(f"Iteration {iteration}: rank(A_tight) = {rank}")
        # Compute direction u in null-space of A_tight
        if len(A1) == 0:
            u = np.random.randn(A.shape[1])
        else:
            ns = null_space(A1)
            if ns.size == 0:
                print("No feasible null-space direction (degenerate or over-constrained).")
                bounded = False
                break
            u = ns[:, 0]
        # Ratio test
        alphas = []
        for bi, ai in zip(b[~mask], A2):
            denom = ai @ u
            if denom > tol:
                alpha = (bi - ai @ z) / denom
                if alpha > tol:
                    alphas.append(alpha)
        if not alphas:
            print(" Detected unbounded region while moving to vertex.")
            bounded = False
            break
        alpha = min(alphas)
        z = z + alpha * u
        mask, A1, A2 = find_tight_rows(A, z, b, tol)
        rank = np.linalg.matrix_rank(A1) if len(A1) else 0
        track_cost.append(float(c @ z))
        track_z.append(z.copy())

    if bounded:
        print(f"Reached vertex after {iteration} steps. rank(A_tight) = {rank}\n")
    else:
        print(f" Phase I stopped early (unbounded or degenerate region). rank(A_tight) = {rank}\n")
    return z, track_cost, track_z, bounded



## **Degeneracy Detection and Handling**

In the simplex algorithm, **degeneracy** occurs when more than *n* constraints become tight (active) at a vertex.  
This can lead to:
- **Zero step sizes** (no progress despite pivoting),
- **Cycling** between the same vertices, or
- **Singular basis matrices (A_B)**.

To handle this, two functions are implemented:

---

#### **1. `is_degenerate(tight)`**
Checks whether the current vertex is **degenerate**, i.e.,  
if the number of tight (active) constraints exceeds the number of variables.



```python
def is_degenerate(tight):
    return tight.shape[0] > tight.shape[1]


## **2: `make_non_degenerate(vector_b_original, epi, reduction_factor=0.5)`**

This function applies a small **ε-perturbation** to the right-hand side (RHS) vector **b**  
to remove **degeneracy** in the simplex algorithm.  
It slightly modifies each constraint boundary so that no two constraints become *exactly equal*,  
ensuring a **unique and non-degenerate vertex**.

---

#### **Code**
```python
def make_non_degenerate(vector_b_original, epi, reduction_factor=0.5):
    epi = epi * reduction_factor
    new_b = np.array([
        vector_b_original[i] + epi ** (i + 1)
        for i in range(len(vector_b_original))
    ])
    return new_b, epi


In [ ]:
def is_degenerate(tight):
    return tight.shape[0] > tight.shape[1]
def make_non_degenerate(vector_b_original, epi, reduction_factor=0.5):
    epi = epi * reduction_factor
    new_b = np.array([
        vector_b_original[i] + epi ** (i + 1)
        for i in range(len(vector_b_original))
    ])
    return new_b, epi


##  Phase II — Vertex-to-Vertex Simplex (Assignment 1)
We now reuse the geometric simplex algorithm from the first assignment to move between vertices until the optimal vertex is found.


In [ ]:
def geometric_simplex_trace(A, b, c, x0, tol=1e-9, epi=1e-3):
    """
    Phase II: Vertex-to-vertex simplex method with degeneracy handling
    using epsilon perturbation (lexicographic method).
    """
    m, n = A.shape
    active_constraints = np.where(np.abs(A @ x0 - b) < tol)[0]
    if len(active_constraints) < n:
        print("Error: initial point not on enough active constraints.")
        return
    B = list(active_constraints[:n])
    iteration = 0

    print("Iter | Vertex (x)           | Obj    | Direction info")
    print("----------------------------------------------------------")

    while True:
        iteration += 1
        A_B = A[B, :]

        # --- Degeneracy handling: Check if A_B is singular
        if np.linalg.matrix_rank(A_B) < n:
            print(f"Degeneracy detected at iteration {iteration}: Singular A_B")
            b, epi = make_non_degenerate(b, epi)
            print("   → Applied epsilon perturbation to b for non-degeneracy")
            A_B = A[B, :]  # Recompute after perturbation

        # Compute feasible directions (columns of -A_B⁻¹)
        try:
            V = -np.linalg.inv(A_B)
        except np.linalg.LinAlgError:
            print(f"Singular A_B even after perturbation at iteration {iteration}.")
            return x0, float(c @ x0)

        directions = [V[:, i] for i in range(len(B))]
        cost_change = [c @ v for v in directions]
        print(f"{iteration:4d} | {np.round(x0,4)} | {c @ x0:7.3f} |", end=" ")

        # --- Check for degeneracy at the current vertex
        tight_mask, A_tight, A_untight = find_tight_rows(A, x0, b, tol)
        if is_degenerate(A_tight):
            print(f"\nDegeneracy detected: {A_tight.shape[0]} tight constraints (> {A_tight.shape[1]})")
            b, epi = make_non_degenerate(b, epi)
            print("  → Applied epsilon perturbation to b for non-degeneracy")
            tight_mask, A_tight, A_untight = find_tight_rows(A, x0, b, tol)

        # --- Optimality check
        if all(val <= tol for val in cost_change):
            print("\nReached optimal vertex.")
            print(f"x* = {np.round(x0,4)}, Objective = {c @ x0:.4f}\n")
            return x0, float(c @ x0)

        # Choose entering variable (first positive cost change)
        i_enter = np.argmax(cost_change)
        v = directions[i_enter]
        print(f"dir {i_enter}, cᵀv = {cost_change[i_enter]:.4f}")

        # --- Ratio test for inactive constraints
        N = [i for i in range(m) if i not in B]
        t_candidates = []
        for j in N:
            a_j = A[j, :]
            denom = a_j @ v
            if denom > tol:
                t = (b[j] - a_j @ x0) / denom
                if t > tol:
                    t_candidates.append((t, j))

        if not t_candidates:
            print("Unbounded in this direction.")
            return x0, np.inf

        t_star, j_enter = min(t_candidates)
        x0 = x0 + t_star * v
        j_leave = B[i_enter]
        B[i_enter] = j_enter
        print(f"   → Move t* = {t_star:.4f} → New vertex {np.round(x0,4)}")
        print(f"     Entering constraint {j_enter}, Leaving constraint {j_leave}")
        print("----------------------------------------------------------")


##  Combine Phase I and Phase II

Now we connect both parts:
1. Move from feasible point to vertex (`feasible_to_vertex`)  
2. Apply the geometric simplex trace (`geometric_simplex_trace`)  


In [ ]:
def simplex_full(filename):
    """
    Complete simplex execution:
    Phase I: Move from feasible point to vertex (check boundedness)
    Phase II: Run vertex-to-vertex simplex if bounded.
    """
    A, b, c, z0 = read_input(filename)
    m, n = A.shape

    # Phase I — move to vertex
    vertex, cost_path, path_z, bounded = feasible_to_vertex(A, b, z0, c, n)

    # If unbounded, stop safely
    if not bounded:
        print("The feasible region is unbounded. Cannot proceed to Phase II (simplex).")
        return vertex, np.inf

    # Phase II — run simplex from that vertex
    x_star, obj = geometric_simplex_trace(A, b, c, vertex)
    print(f"Final Solution: x* = {np.round(x_star,4)}, Objective = {obj:.4f}")
    return x_star, obj


## Testing Simplex on Multiple Cases

We’ll now create a set of diverse linear programming test cases:

| File | Description | Expected Result |
|------|--------------|----------------|
| `tc1.csv` | Normal bounded LP | Finite optimum |
| `tc2.csv` | Degenerate vertex | Finite optimum, multiple tight constraints |
| `tc3.csv` | Unbounded LP | Reports “unbounded” |
| `tc4.csv` | Non-vertex feasible start | Moves to vertex first |
| `tc5.csv` | 3D bounded LP | Finite optimum in higher dimension |

Each file will be saved and then solved using the full simplex pipeline.


In [ ]:
import pandas as pd

# ---------- Testcase 1: Normal bounded LP ----------
# max z = 3x1 + 2x2
# s.t. 2x1 + x2 <= 10, x1 + 3x2 <= 15, x1 <= 6, x1,x2 >= 0
tc1 = [
    [2, 2, ""],    # initial feasible point
    [3, 2, ""],
    [2, 1, 10],
    [1, 3, 15],
    [1, 0, 6],
    [-1, 0, 0],
    [0, -1, 0]
]
pd.DataFrame(tc1).to_csv("tc1.csv", header=False, index=False)

# ---------- Testcase 2: Degenerate vertex ----------
# max z = 3x1 + 2x2
# s.t. x1 + x2 <= 4, x1 <= 2, x2 <= 2, x1,x2 >= 0
tc2 = [
    [1, 1, ""],
    [3, 2, ""],
    [1, 1, 4],
    [1, 0, 2],
    [0, 1, 2],
    [-1, 0, 0],
    [0, -1, 0]
]
pd.DataFrame(tc2).to_csv("tc2.csv", header=False, index=False)

# ---------- Testcase 3: Unbounded LP ----------
# max z = 3x1 + 4x2
# s.t. x1 - x2 <= 2, x1 >= 0, x2 >= 0
tc3 = [
    [1, 1, ""],
    [3, 4, ""],
    [1, -1, 2],
    [-1, 0, 0],
    [0, -1, 0]
]
pd.DataFrame(tc3).to_csv("tc3.csv", header=False, index=False)

# ---------- Testcase 4: Feasible point not at vertex ----------
# max z = 5x1 + 3x2
# s.t. x1 + x2 <= 8, x1 <= 5, x2 <= 6, x1,x2 >= 0
tc4 = [
    [2, 3, ""],
    [5, 3, ""],
    [1, 1, 8],
    [1, 0, 5],
    [0, 1, 6],
    [-1, 0, 0],
    [0, -1, 0]
]
pd.DataFrame(tc4).to_csv("tc4.csv", header=False, index=False)

# ---------- Testcase 5: 3D bounded LP ----------
# max z = 4x1 + 3x2 + 5x3
# s.t. 2x1 + x2 + x3 <= 10, x1 + 3x2 + 2x3 <= 15, x1 + x2 + 4x3 <= 18, x1,x2,x3 >= 0
tc5 = [
    [1, 1, 1, ""],
    [4, 3, 5, ""],
    [2, 1, 1, 10],
    [1, 3, 2, 15],
    [1, 1, 4, 18],
    [-1, 0, 0, 0],
    [0, -1, 0, 0],
    [0, 0, -1, 0]
]
pd.DataFrame(tc5).to_csv("tc5.csv", header=False, index=False)



## Run the Simplex Algorithm on All Test Cases
Now we will execute the full simplex pipeline (Phase I + Phase II)  
on each of the generated files and print the results.


In [ ]:
test_files = [f"tc{i}.csv" for i in range(1, 6)]
results = []

for f in test_files:
    print(f"\n================ Running {f} ================")
    x_star, obj = simplex_full(f)
    results.append((f, np.round(x_star, 4).tolist(), obj))




================ Running tc1.csv ================
Phase I — Moving from feasible point to vertex...
Iteration 1: rank(A_tight) = 0
Iteration 2: rank(A_tight) = 1
Reached vertex after 2 steps. rank(A_tight) = 2

Iter | Vertex (x)           | Obj    | Direction info
----------------------------------------------------------
   1 | [3. 4.] |  17.000 | 
Reached optimal vertex.
x* = [3. 4.], Objective = 17.0000

Final Solution: x* = [3. 4.], Objective = 17.0000

================ Running tc2.csv ================
Phase I — Moving from feasible point to vertex...
Iteration 1: rank(A_tight) = 0
Iteration 2: rank(A_tight) = 1
Reached vertex after 2 steps. rank(A_tight) = 2

Iter | Vertex (x)           | Obj    | Direction info
----------------------------------------------------------
   1 | [0. 2.] |   4.000 | dir 1, cᵀv = 3.0000
   → Move t* = 2.0000 → New vertex [2. 2.]
     Entering constraint 0, Leaving constraint 3
----------------------------------------------------------
   2 | [2. 2.] 

In [ ]:
test_files = [f"testcase{i}.csv" for i in range(1, 5)]
results = []

for f in test_files:
    print(f"\n================ Running {f} ================")
    x_star, obj = simplex_full(f)
    results.append((f, np.round(x_star, 4).tolist(), obj))




================ Running testcase1.csv ================
Phase I — Moving from feasible point to vertex...
Iteration 1: rank(A_tight) = 0
Iteration 2: rank(A_tight) = 1
Reached vertex after 2 steps. rank(A_tight) = 2

Iter | Vertex (x)           | Obj    | Direction info
----------------------------------------------------------
   1 | [1. 1.] |   7.000 | 
Reached optimal vertex.
x* = [1. 1.], Objective = 7.0000

Final Solution: x* = [1. 1.], Objective = 7.0000

================ Running testcase2.csv ================
Phase I — Moving from feasible point to vertex...
Iteration 1: rank(A_tight) = 0
Iteration 2: rank(A_tight) = 1
 Detected unbounded region while moving to vertex.
 Phase I stopped early (unbounded or degenerate region). rank(A_tight) = 1

The feasible region is unbounded. Cannot proceed to Phase II (simplex).

================ Running testcase3.csv ================
Phase I — Moving from feasible point to vertex...
Iteration 1: rank(A_tight) = 0
Iteration 2: rank(A_tight) =